# 04 · LoRA fine-tune — Qwen3-ASR-1.7B on three large Vietnamese corpora

Fine-tunes `Qwen/Qwen3-ASR-1.7B-hf` with a **PEFT LoRA** adapter on the combined
training splits of viVoice, VietSpeech and VieNeu-TTS-140h, then reports
**before/after WER on each corpus's own held-out test split**, diffed against
`results/3ds_baseline_metrics.json` from `03_eval_baseline_3ds.ipynb`.

**Run notebook 3 first.** It builds the caches this notebook trains on and
writes the baseline these numbers are compared against.

## Scale: 100 h per corpus, 300 h total

`HOURS_PER_CORPUS = 100.0` in section 1 — an equal-hours cap on each of the
three, not the whole of any of them. That is ~35 GB on disk, ~4 wall-hours to
download, and ~5 h for one training epoch at batch 8. Set it to `None` for the
full ~2,240 h once this run has shown the recipe works; that is ~258 GB, ~30
wall-hours of download and ~33 h of training, so it is not a knob to turn
speculatively.

Capping each corpus at the same number of hours also balances the mixture. The
corpora are wildly unequal in size (viVoice ~1,000 h, VietSpeech ~1,100 h,
VieNeu 140.7 h); taking them in full would let the two big ones set the gradient
almost entirely, and the equal cap is what keeps VieNeu's clean read speech from
being rounded away.

## What is different from `02_lora_finetune.ipynb`

Notebook 2 trains on a ~34 h blend of four sources and holds out slices of
*public benchmarks* (VIVOS, Common Voice, VLSP). Here each corpus keeps its own
train/val/test, so the question is per-corpus: does more data from these three
sources move each one's held-out WER. Every test split is speaker- or
recording-disjoint from training, so **all three numbers stay genuinely
held-out** — unlike notebook 2's section 9, where the benchmarks become
in-domain after the fine-tune.

The recipe itself is carried over from notebook 2 run 2, which is the last
configuration known to behave: LoRA on decoder attention **and** MLP
(17.4 M trainable, 0.85%), lr `1e-4`, effective batch 16, bf16, gradient
checkpointing. The only change is *how* that effective batch is reached — batch
8 × grad-accum 2 instead of 1 × 16, which is 7x the throughput for the same
optimizer step, and is what makes 300 h finish in an evening.

## A note on transcript style

The three corpora do not agree, and the model fits whatever it is shown:

| corpus | casing | punctuation |
|---|---|---|
| `vivoice_full` | mixed case | yes |
| `vietspeech` | all lowercase | **none** |
| `vieneu` | mixed case | yes |

`vi_norm` lowercases and strips punctuation before scoring, so this costs model
capacity but not the metric — the same trade `mixture.py` documents for VIVOS
and VLSP. Worth watching anyway: if VietSpeech dominates the mixture the model
may stop emitting punctuation altogether, which is invisible to WER but plainly
visible in the prediction CSVs.

`mixture.normalize_train_text` runs on ingest, but it only folds ALL-CAPS rows —
none of these three is ALL-CAPS, so it is a no-op here. The run-1 casing bug it
exists to prevent is documented in `mixture.py`.

## The corpora are very unequal in difficulty too

VieNeu is clean studio read speech; viVoice is YouTube; VietSpeech is social
media across three regional accents. Expect a much lower baseline WER on VieNeu,
and read its delta with that in mind — little headroom means a small change
there says less than the same change on VietSpeech. Section 9's per-corpus
breakdown exists precisely so these do not get averaged into one number.

**16 GB VRAM budget:** batch size 8 · gradient accumulation 2 · gradient
checkpointing · bf16 · `adamw_torch_fused`. Measured peak at batch 8 was ~7.6 GB.

Run top-to-bottom (Kernel → Restart & Run All).

In [ ]:
import os, random, numpy as np, torch
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (see .env.example)"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1 · Load the corpora and assemble the training mixture

Cache-aware and identical to notebook 3, so this is a no-op if that ran first.
`HOURS_PER_CORPUS` and `NAMES` must match notebook 3's values, or the two
notebooks build different splits and the before/after comparison is meaningless.
Section 9 prints a warning if the scored `n` disagrees, which is the symptom.

In [ ]:
import corpora

# Hours streamed per corpus; None = the whole corpus.
#
# 100 h x 3 = 300 h. Measured costs on this box, so this is a budget decision
# rather than a guess:
#   disk      115 MB per audio-hour of 16 kHz PCM_16 wav  ->  ~35 GB
#   download  74 audio-hours per wall-hour sustained      ->  ~4 h to build
#
# Raising this to None takes all three in full (~2,240 h): ~258 GB and ~30
# wall-hours to download, then ~10x this notebook's epoch time. Start here.
#
# Each corpus is capped independently, so the mixture is balanced by hours even
# though the corpora are not: viVoice ~1,000 h, VietSpeech ~1,100 h, VieNeu
# 140.7 h. At 100 h each, VieNeu contributes ~71% of itself while the other two
# contribute ~10% — deliberate, so no single source dominates the gradient.
#
# Note for VieNeu: its 193 voices sit contiguously across 49 shards, so hours
# buy clips much faster than they buy speakers — measured, 0.4 h gave 4 voices
# and 3 h gave 5. 100 h is ~35 of the 49 shards, so expect ~130-150 voices,
# which is enough for a speaker-disjoint split to mean something.
HOURS_PER_CORPUS = 100.0
# "vivoice_full" streams viVoice as its native clips and honours the cap above.
# The other entry, "vivoice", is the 8.4 h merged cache notebooks 1-2 use — it
# ignores HOURS_PER_CORPUS entirely, so swap it in only for a quick run whose
# viVoice number is directly comparable to notebook 1.
NAMES = ["vivoice_full", "vietspeech", "vieneu"]

# Probes access first and skips what it cannot reach, so an unapproved gated
# corpus costs a printed line rather than a crash an hour into the build.
built = corpora.prepare_all(NAMES, target_hours=HOURS_PER_CORPUS)
AVAILABLE = list(built)
print("\nusable corpora:", AVAILABLE)

In [ ]:
from datasets import concatenate_datasets

VAL_LIMIT = 200   # validation clips total; eval runs at batch size 1, so keep it small

train = corpora.load_mixture(AVAILABLE, "train")
val = concatenate_datasets(
    [corpora.eval_rows(n, VAL_LIMIT // max(1, len(AVAILABLE)), split="val", seed=SEED)
     for n in AVAILABLE])

# Integer length in ms, for TrainingArguments(group_by_length=True). Trainer
# reads this column directly; without it, LengthGroupedSampler would try to
# measure length from `input_ids`, which this dataset does not carry.
train = train.add_column("length", [int(d * 1000) for d in train["duration"]])

print({"train": len(train), "val": len(val)})
print()
import mixture
summary = mixture.summarize(list(train))
print(summary.to_string(index=False))

hours = summary["hours"].sum()
print(f"\ntotal {hours:.0f} h of audio across {len(train)} examples")
for bs, rate in ((1, 1.42), (4, 5.10), (8, 9.82), (16, 12.22)):
    print(f"  batch {bs:>2}: {rate:5.2f} ex/s -> {len(train) / rate / 3600:6.1f} h per epoch")

## 2 · Duration profile

Worth a look before training. VietSpeech is mostly 2–6 s and viVoice is 5–60 s,
so the combined mixture is bimodal. If one corpus dominates a bucket, the
fine-tune is effectively adapting to that corpus's clip length — and each
corpus is scored on its *own* length distribution in section 7.

In [ ]:
import pandas as pd

prof = pd.crosstab(pd.Series(train["source"], name="source"),
                   pd.Series(train["bucket"], name="bucket"), margins=True)
print(prof.to_string())
prof

## 3 · Load the base model + processor

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map=DEVICE,
)
print("loaded", type(model).__name__, model.dtype, model.device)

## 4 · Attach the LoRA adapter (decoder attention + MLP)

`q/k/v_proj` exist in both the audio encoder and the text decoder, so we match
**only** the `model.language_model` projections with a regex — the audio encoder
stays frozen. MLP projections are included alongside attention: the decoder MLP
is where a transformer stores most of its lexical knowledge, which is what
adapting to a language's vocabulary needs.

In [ ]:
from peft import LoraConfig, get_peft_model

# Full-match regex over module names -> only the 28 decoder layers, never the
# audio encoder. 4 attention + 3 MLP projections per layer = 196 total.
TARGET_RE = (r"model\.language_model\.layers\.\d+\."
             r"(self_attn\.(q_proj|k_proj|v_proj|o_proj)"
             r"|mlp\.(gate_proj|up_proj|down_proj))")

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()   # required for grad checkpointing + PEFT

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=TARGET_RE,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

adapted = [n for n, _ in model.named_modules() if n.endswith("lora_A.default")]
assert len(adapted) == 28 * 7, f"expected 196 LoRA-adapted projections, got {len(adapted)}"
# The audio encoder has q/k/v_proj and fc1/fc2 of its own, so a regex that
# widened by accident would silently unfreeze it — and the encoder is exactly
# what this run means to hold fixed.
assert not [n for n in adapted if ".language_model." not in n], \
    "LoRA leaked outside model.language_model — audio encoder must stay frozen"
print("LoRA-adapted projections:", len(adapted))

## 5 · Data collator (on-the-fly)

For each raw example we build the ASR request (audio + prompt), then append the
target transcription tokens + EOS as **labels**, masking the prompt with `-100`.
Features are computed on the fly rather than serialized to disk.

In [ ]:
import data_prep

EOS_ID = processor.tokenizer.eos_token_id
PAD_ID = processor.tokenizer.pad_token_id

def collate(features):
    ids_list, lbl_list, feats, feat_masks = [], [], [], []
    for ex in features:
        arr, _ = data_prep.read_audio(ex["audio_path"])   # 16 kHz mono
        req = processor.apply_transcription_request(audio=arr, language="Vietnamese")
        p_ids = req["input_ids"][0]
        tgt = processor.tokenizer(ex["text"], add_special_tokens=False,
                                  return_tensors="pt")["input_ids"][0]
        tgt = torch.cat([tgt, torch.tensor([EOS_ID], dtype=tgt.dtype)])
        ids = torch.cat([p_ids, tgt])
        lbl = torch.cat([torch.full((len(p_ids),), -100, dtype=torch.long), tgt])
        ids_list.append(ids); lbl_list.append(lbl)
        feats.append(req["input_features"][0])
        feat_masks.append(req["input_features_mask"][0])

    maxlen = max(len(x) for x in ids_list)
    def pad1(x, val):
        return torch.cat([x, torch.full((maxlen - len(x),), val, dtype=x.dtype)])
    input_ids = torch.stack([pad1(x, PAD_ID) for x in ids_list])
    labels = torch.stack([pad1(x, -100) for x in lbl_list])
    attn = torch.stack([
        torch.cat([torch.ones(len(x), dtype=torch.long),
                   torch.zeros(maxlen - len(x), dtype=torch.long)]) for x in ids_list])

    maxT = max(f.shape[-1] for f in feats)
    input_features = torch.stack(
        [torch.cat([f, torch.zeros(f.shape[0], maxT - f.shape[-1], dtype=f.dtype)], dim=-1)
         for f in feats])
    input_features_mask = torch.stack(
        [torch.cat([m, torch.zeros(maxT - len(m), dtype=m.dtype)]) for m in feat_masks])

    return {"input_ids": input_ids, "attention_mask": attn, "labels": labels,
            "input_features": input_features, "input_features_mask": input_features_mask}

# sanity check on 2 examples
_b = collate([train[0], train[1]])
print({k: tuple(v.shape) for k, v in _b.items()})

## 6 · Training configuration

In [ ]:
from transformers import Trainer, TrainerCallback, TrainingArguments

LOG_EVERY = 50
EPOCHS = 1

# Batch size is THE decision at this scale. From the throughput table in the
# README (measured on this box):
#     batch  1 ->  1.42 ex/s   4.78 GB      batch  8 ->  9.82 ex/s  ~7.6 GB
#     batch  4 ->  5.10 ex/s   6.00 GB      batch 16 -> 12.22 ex/s  10.78 GB
# The 300 h mixture is roughly 230k clips, ~185k of them in the train split. At
# batch 1 that is 36 h/epoch; at batch 8 it is ~5 h. Accumulation drops to 2 so
# the effective batch stays 16, i.e. the run-2 recipe is unchanged and only
# throughput moves. Drop to 4x4 if you OOM (the 60 s clips are the risk, not
# the mean). The cell above prints the real per-epoch estimate from len(train).
TRAIN_BATCH = 8
GRAD_ACCUM = 2

# ceil, matching how Trainer counts: the trailing partial accumulation group
# still fires an optimizer step.
per_step = TRAIN_BATCH * GRAD_ACCUM
steps_per_epoch = -(-len(train) // per_step)
total_steps = int(steps_per_epoch * EPOCHS)
# Sample the validation curve often enough to catch a divergence before it costs
# hours. Quartiles alone would be ~75 min apart at this size.
EVAL_EVERY = min(2000, max(50, total_steps // 8))


class LossPrinter(TrainerCallback):
    """Print train and validation loss on their own lines.

    Needed because transformers' notebook progress table only writes a
    training-loss row when ``eval_strategy == "no"`` (see on_log in
    transformers/utils/notebook.py). With eval enabled the table is written
    only at eval time, so without this the first loss shown is at EVAL_EVERY.
    """

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        step = f"{state.global_step:5d}/{state.max_steps}"
        if "loss" in logs:          # training log, every logging_steps
            print(f"[train] {step}  loss {logs['loss']:.4f}"
                  f"  lr {logs.get('learning_rate', float('nan')):.2e}", flush=True)
        if "eval_loss" in logs:     # evaluation log, every eval_steps
            print(f"[ val ] {step}  loss {logs['eval_loss']:.4f}"
                  f"  ({logs.get('eval_runtime', 0):.0f}s)", flush=True)


args = TrainingArguments(
    output_dir="checkpoints/vi_lora_3ds",
    per_device_train_batch_size=TRAIN_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    # Tuned for the 17.4M-param attention+MLP surface (2.7x run 1's). The same
    # step size over more directions moves the decoder's lexical weights further
    # than intended, and those weights hold the language-model prior.
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=LOG_EVERY,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY,
    save_strategy="steps",
    save_steps=EVAL_EVERY,          # must equal eval_steps for load_best_model_at_end
    # Make the val split actually do its job: keep the checkpoint that validated
    # best rather than whichever ran last.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,             # best + latest; 1 can delete the best
    per_device_eval_batch_size=4,   # 16 GB: the default 8 OOMs on 60 s clips
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,    # our collator consumes the raw columns
    # The collator decodes a wav and computes a mel spectrogram per example, so
    # at ~10 ex/s the loader is doing real work. 2 workers fed batch 1; batch 8
    # needs more or the GPU waits on the CPU.
    dataloader_num_workers=6,
    dataloader_pin_memory=True,
    # The mixture is bimodal — VietSpeech is 2-6 s, viVoice 5-60 s. Batching
    # those together pads every short clip out to the longest in the batch and
    # wastes most of the compute. This groups similar lengths into a batch.
    # (transformers v5 renamed this from the old `group_by_length=True`.)
    train_sampling_strategy="group_by_length",
    length_column_name="length",
)
trainer = Trainer(model=model, args=args, train_dataset=train,
                  eval_dataset=val, data_collator=collate,
                  callbacks=[LossPrinter()])

print(f"steps/epoch {steps_per_epoch}  total updates {total_steps}  "
      f"eval every {EVAL_EVERY}  train-loss lines {total_steps // LOG_EVERY}")

## 7 · Train

Watch VRAM stay < 16 GB — measured peak on this box was 7.09 GB, and
`corpora.MAX_CLIP_S` drops anything over 60 s at build time. If you do OOM,
raise `gradient_accumulation_steps`.

Watch `eval_loss`: if it is still falling at the end the run is undertrained and
a second epoch is justified; if it turns upward, `load_best_model_at_end` hands
back the best checkpoint rather than the last one, so the final adapter may not
be the one trained longest.

In [ ]:
trainer.train()
trainer.save_model("checkpoints/vi_lora_3ds")   # saves the LoRA adapter
print("peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 2))
print("adapter saved -> checkpoints/vi_lora_3ds")

## 8 · Evaluate on each corpus's held-out test split

Same `corpora.score_corpus` call as notebook 3, with the same `EVAL_LIMIT` and
`SEED` — so the two runs score identical clips and the delta is real.

In [ ]:
import json, pathlib
import pandas as pd

EVAL_LIMIT = 500   # must match notebook 3
BATCH_SIZE = 8

model.config.use_cache = True
model.eval()
processor.tokenizer.padding_side = "left"   # required for correct batched generation

lora_metrics, lora_frames = {}, {}
for name in AVAILABLE:
    print(f"\n=== {name} · {corpora.corpus_dir(name)} [test] ===")
    m, f = corpora.score_corpus(model, processor, name, limit=EVAL_LIMIT,
                                batch_size=BATCH_SIZE, seed=SEED)
    lora_metrics[name], lora_frames[name] = m, f
    print(f"   WER={m['wer']:.4f}  CER={m['cer']:.4f}  "
          f"(legacy WER={m['wer_legacy']:.4f})  n={m['n']}")

pathlib.Path("results").mkdir(exist_ok=True)
for name, f in lora_frames.items():
    f.to_csv(f"results/3ds_lora_{name}_predictions.csv", index=False)
with open("results/3ds_lora_metrics.json", "w", encoding="utf-8") as fh:
    json.dump(lora_metrics, fh, ensure_ascii=False, indent=2)
print("\nsaved -> results/3ds_lora_metrics.json")

## 9 · Before / after

Every row here is speaker- or recording-disjoint from training, so these are
genuine held-out numbers — not the in-domain benchmark deltas of notebook 2's
section 9.

Read the `n` column before the deltas: at n=500 a WER difference under about
half a point is inside sampling noise. Section 10 puts a bootstrap interval
around it.

In [ ]:
try:
    base_metrics = json.load(open("results/3ds_baseline_metrics.json"))
except FileNotFoundError:
    base_metrics = {}
    print("No results/3ds_baseline_metrics.json — run 03_eval_baseline_3ds.ipynb "
          "for the baseline.")

rows = []
for name, l in lora_metrics.items():
    b = base_metrics.get(name)
    if b and b.get("n") != l.get("n"):
        print(f"WARNING {name}: baseline scored n={b['n']} but this run scored "
              f"n={l['n']}. EVAL_LIMIT/SEED/HOURS_PER_CORPUS must match "
              f"notebook 3 — the row below is not a valid comparison.")
    rows.append({
        "corpus": name,
        "n": l["n"],
        "baseline WER %": round(100 * b["wer"], 2) if b else None,
        "LoRA WER %": round(100 * l["wer"], 2),
        "abs delta": round(100 * (l["wer"] - b["wer"]), 2) if b else None,
        "rel %": round(100 * (l["wer"] - b["wer"]) / b["wer"], 1) if b else None,
        "held-out?": "yes — speaker/recording-disjoint",
    })
cmp = pd.DataFrame(rows)
print()
print(cmp.to_string(index=False))
cmp

## 10 · Is the change real?

A WER delta on a few hundred clips is easy to over-read. This resamples the
per-clip errors to put a 95% interval around each corpus's change: if the
interval spans zero, the run did not measurably move that corpus.

Notebook 2's run 1 is the cautionary case — a 0.02 point "improvement" on
viVoice that turned out to be 2 word errors in 10,155, with a CI of
[-0.42%, +0.37%].

This needs notebook 3's per-clip predictions, so load those CSVs first.

In [ ]:
base_frames_available = {}
for name in AVAILABLE:
    p = f"results/3ds_baseline_{name}_predictions.csv"
    try:
        base_frames_available[name] = pd.read_csv(p)
    except FileNotFoundError:
        print(f"missing {p} — run notebook 3 to enable the significance test for {name}")
print("loaded baseline frames for:", list(base_frames_available))

In [ ]:
import numpy as np
from vi_norm import normalize_vi
import jiwer

def _clip_counts(refs, hyps):
    """Per-clip (errors, reference words) under the shared normalizer."""
    out = []
    for r, h in zip(refs, hyps):
        r, h = normalize_vi(r), normalize_vi(h)
        if not r.strip():
            continue
        m = jiwer.process_words(r, h)
        out.append((m.substitutions + m.deletions + m.insertions, len(r.split())))
    return np.array(out, dtype=float)

rng = np.random.default_rng(SEED)
for name in lora_metrics:
    if name not in base_frames_available:
        continue
    b = _clip_counts(base_frames_available[name]["ref"], base_frames_available[name]["hyp"])
    l = _clip_counts(lora_frames[name]["ref"], lora_frames[name]["hyp"])
    n = min(len(b), len(l))
    deltas = []
    for _ in range(2000):
        idx = rng.integers(0, n, n)
        deltas.append(l[idx, 0].sum() / l[idx, 1].sum() - b[idx, 0].sum() / b[idx, 1].sum())
    lo, hi = np.percentile(deltas, [2.5, 97.5]) * 100
    point = (l[:n, 0].sum() / l[:n, 1].sum() - b[:n, 0].sum() / b[:n, 1].sum()) * 100
    verdict = "not distinguishable from noise" if lo < 0 < hi else "significant"
    print(f"{name:<12} delta {point:+.3f} pts  95% CI [{lo:+.2f}, {hi:+.2f}]  -> {verdict}")